# Pipeline: Download → Normalize → Analyze

Orchestrator notebook that runs the full data pipeline using modules from `src/`.

Each step can be skipped if the data already exists.

In [ ]:
import os
import sys

# Install dependencies
!pip install -q pyspark==3.4.2 apache-sedona==1.6.1 requests folium

## Step 1: Download Raw Data

Downloads NY crime data (Socrata API), police stations, and transport stations (Overpass API).

Skips download if files already exist.

In [ ]:
from src.config import NY_CRIMES_RAW, NY_POLICE_STATIONS, NY_TRANSPORT_STATIONS
from src.download import download_ny_crimes, download_police_stations, download_transport_stations

# Check what already exists
crimes_exist = os.path.isdir(NY_CRIMES_RAW) and len(os.listdir(NY_CRIMES_RAW)) > 0
police_exist = os.path.isfile(NY_POLICE_STATIONS)
transport_exist = os.path.isfile(NY_TRANSPORT_STATIONS)

if crimes_exist:
    print(f"Crime CSVs already exist in {NY_CRIMES_RAW} — skipping download")
else:
    download_ny_crimes()

if police_exist:
    print(f"Police stations already exist at {NY_POLICE_STATIONS} — skipping download")
else:
    download_police_stations()

if transport_exist:
    print(f"Transport stations already exist at {NY_TRANSPORT_STATIONS} — skipping download")
else:
    download_transport_stations()

print("\nStep 1 complete.")

## Step 2: Normalize Crime Data

Renames columns, casts types, filters invalid coordinates, and saves as partitioned parquet.

Skips if normalized parquet already exists.

In [ ]:
from src.config import NY_CRIMES_NORMALIZED
from src.spark_session import get_or_create_spark
from src.normalize import normalize_ny_crimes

spark = get_or_create_spark()

if os.path.isdir(NY_CRIMES_NORMALIZED):
    print(f"Normalized parquet already exists at {NY_CRIMES_NORMALIZED} — skipping normalization")
    df_crimes = spark.read.parquet(NY_CRIMES_NORMALIZED)
else:
    df_crimes = normalize_ny_crimes(spark)

print(f"Crimes loaded: {df_crimes.count():,} records")
df_crimes.printSchema()
df_crimes.show(5)

## Step 3: Spatial Analysis

Loads police/transport stations, computes distances from each crime to nearest station, and categorizes proximity.

In [ ]:
from sedona.register import SedonaRegistrator
from pyspark.sql.functions import col, expr

SedonaRegistrator.registerAll(spark)

# Load station data
df_police = spark.read.option("header", True).csv(NY_POLICE_STATIONS)
df_transport = spark.read.option("header", True).csv(NY_TRANSPORT_STATIONS)

print(f"Police stations: {df_police.count()}")
print(f"Transport stations: {df_transport.count()}")

In [ ]:
# Create geometry columns using Sedona
df_crimes_geo = df_crimes.withColumn(
    "geometry", expr("ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE))")
)
df_police_geo = df_police.withColumn(
    "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)
df_transport_geo = df_transport.withColumn(
    "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)

In [ ]:
from src.analyze import compute_spatial_distances, categorize_proximity

# Compute distances to nearest police and transport stations
df_distances = compute_spatial_distances(spark, df_crimes_geo, df_police_geo, df_transport_geo)

# Categorize proximity
df_categorized = categorize_proximity(df_distances)

print("Proximity categorization:")
df_categorized.select("offense_description", "dist_to_nearest_police_km", "near_police",
                      "dist_to_nearest_transport_km", "near_transport").show(10)

## Step 4: Temporal Analysis

Enrich crime data with time features (hour, day of week, month, time slot) and analyze temporal patterns.

In [ ]:
from src.analyze import add_temporal_columns, analyze_temporal_patterns, analyze_crime_type_by_time_slot

# Enrich crimes with temporal features
df_temporal = add_temporal_columns(df_crimes)

# Analyze patterns
crimes_by_hour, crimes_by_day, crimes_by_month, crimes_by_slot = analyze_temporal_patterns(df_temporal)

print("Crimes by hour of day:")
crimes_by_hour.show(24)

print("Crimes by day of week (1=Domingo, 2=Lunes, ..., 7=Sabado):")
crimes_by_day.show(7)

print("Crimes by month:")
crimes_by_month.show(12)

print("Crimes by time slot:")
crimes_by_slot.show()

print("Top crime types by time slot:")
analyze_crime_type_by_time_slot(df_temporal).show(30, truncate=False)

## Step 5: FP-Growth (Association Rules)

In [ ]:
from src.analyze import run_fp_growth

freq_itemsets, rules = run_fp_growth(df_categorized)

print("Frequent Itemsets:")
freq_itemsets.orderBy("freq", ascending=False).show(20, truncate=False)

print("Association Rules:")
rules.orderBy("confidence", ascending=False).show(20, truncate=False)

## Step 6: Expanded FP-Growth (with Time Slot and Demographics)

Adds time slot (MADRUGADA/MANANA/TARDE/NOCHE) and victim demographics to the association rules for richer pattern discovery.

## Step 7: K-Means Clustering

## Step 5: K-Means Clustering

## Step 8: Visualization

## Step 6: Visualization

In [ ]:
import folium

# Create map centered on New York
ny_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11)

# Add cluster centers
for i, center in enumerate(model.clusterCenters()):
    folium.Marker(
        location=[center[1], center[0]],  # lat, lon
        popup=f"Cluster {i}",
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(ny_map)

# Add police stations
police_rows = df_police.collect()
for row in police_rows:
    if row["lat"] and row["lon"]:
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=4,
            color="blue",
            fill=True,
            popup=row["name"] or "Police Station",
        ).add_to(ny_map)

# Add transport stations (sample to avoid overloading map)
transport_rows = df_transport.limit(200).collect()
for row in transport_rows:
    if row["lat"] and row["lon"]:
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=3,
            color="green",
            fill=True,
            popup=row["name"] or "Transport Station",
        ).add_to(ny_map)

ny_map

In [ ]:
spark.stop()
print("Pipeline complete.")